## FInetuning TinyLlama model with mdeical data using hugging Face ecosystem

*  model used - https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0

*  Dataset Used - https://huggingface.co/datasets/aboonaji/wiki_medical_terms_llam2_format




### Imports

In [15]:
pip install transformers datasets peft bitsandbytes accelerate trl

In [16]:
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import LoraConfig

from trl import SFTTrainer, SFTConfig

### Loading the model

In [17]:
from transformers import AutoModelForCausalLM
modelName='TinyLlama/TinyLlama-1.1B-Chat-v1.0'

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=modelName,quantization_config=bnb_config)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### Loading Tokenizer

In [19]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=modelName, use_fast=True)

tokenizer.pad_token = tokenizer.eos_token


In [20]:
text = "Patient has diabetes."
tokens = tokenizer(text)
print(tokens)

{'input_ids': [1, 4121, 993, 756, 652, 370, 10778, 29889], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


### Test before fine tuning

In [21]:
from transformers import pipeline

def getAnswer():
  pipe = pipeline("text-generation", model=modelName)
  messages = [
    {"role": "user", "content": "What is Paracetamol poisoning Explain"},
  ]
  return pipe(messages)

In [22]:
getAnswer()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'generated_text': [{'role': 'user',
    'content': 'What is Paracetamol poisoning Explain'},
   {'role': 'assistant',
    'content': "Paracetamol (also known as Tylenol) is a non-steroidal anti-inflammatory drug (NSAID) that is used to relieve pain, fever, and inflammation. It works by blocking the activity of enzymes in the body that cause inflammation and pain.\n\nParacetamol poisoning, also called acetaminophen poisoning, is a medical emergency that can lead to serious health problems. The most common symptoms of Paracetamol poisoning include nausea, vomiting, diarrhea, abdominal pain, and drowsiness. In severe cases, Paracetamol poisoning can lead to coma, seizures, and even death.\n\nThe most common way to acquire Paracetamol poisoning is through intentional self-harm (Suicide) or accidental overdose (poisoning). Paracetamol overdose can occur when too much Paracetamol is taken in a short period of time, leading to severe toxicity.\n\nParacetamol poisoning can be treated by imme

### Setting Training Arguments

In [23]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./tinyllama-medical",
    per_device_train_batch_size=4,
    max_steps=100
)

### Load dataset

In [24]:
from datasets import load_dataset

dataset = load_dataset("aboonaji/wiki_medical_terms_llam2_format",split='train')
print(len(dataset))

print(dataset[0])

dataset = dataset.select(range(1000))
print(len(dataset))


6861
{'text': '<s> [INST] <<SYS>> You are a helpful, respectful, and honest assistant. Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature. If a question does not make any sense or is not factually coherent, explain why instead of answering something not correct. If you don\'t know the answer to a question, please don\'t share false information. <</SYS>> What is Paracetamol poisoning and explain in detail? [/INST] Paracetamol poisoning, also known as acetaminophen poisoning, is caused by excessive use of the medication paracetamol (acetaminophen). Most people have few or non-specific symptoms in the first 24 hours following overdose. These include feeling tired, abdominal pain, or nausea. This is typically followed by a couple of days without any symptoms, after which yellowish skin, blood clot

### Configure LoRA


1.   Instead of changing all model weights
2.   we only train tiny adapter matrices.

3. q_proj & v_proj are attention projection layers.

Changing them is enough to adapt the model effectively




In [25]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

### Create Supervised Fine Tuning Trainer

In [26]:
training_args = SFTConfig(
    output_dir="./tinyllama-medical",

    learning_rate=2e-4,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=8,

    num_train_epochs=3,

    logging_steps=10,

    save_steps=200,

    max_length=1024,

    fp16=True,

    report_to="none",
    packing=True

)

In [27]:
trainer = SFTTrainer(
    model=model,

    args=training_args,

    train_dataset=dataset,

    processing_class=tokenizer,

    peft_config=peft_config,
)

### Start training

In [ ]:
trainer.train()

### Saved finetuned model
What gets saved:
LoRA adapter weights
tokenizer configuration
training metadata

The base TinyLlama model is not duplicated.

In [ ]:
trainer.save_model("./tinyllama-medical")

tokenizer.save_pretrained("./tinyllama-medical")

### Check after fine tuning

In [ ]:
getAnswer()